# 🕌 Bulaq 1280 AH ByT5 Arabic OCR Corrector — v2 (Sentence-Level)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefbouhaik/bulaq-ocr-transformer/blob/main/Train_Bulaq_Transformer_Colab.ipynb)

Fine-tunes **`google/byt5-small`** on **69,640 sentence-level** Bulaq 1280 AH OCR correction pairs.

### ⚡ v2 Fixes (from failed v1)
| Problem | v1 (Failed) | v2 (Fixed) |
|:---|:---|:---|
| Dataset granularity | 60% single-word pairs | Full sentence pairs (15-120 chars) |
| Punctuation-only pairs | 32% of dataset | < 3% of dataset |
| Learning rate | 1e-3 (gradient explosion) | 3e-4 (stable convergence) |
| Dataset size | 28,468 word fragments | 69,640 sentence pairs |
| Training signal | Add comma / fix one letter | Multi-error sentence healing |

## 1. Check Hardware Acceleration
*Works on TPU v5e, A100 GPU, L4 GPU, or CPU (slower)*

In [ ]:
# Check for GPU
!nvidia-smi 2>/dev/null || echo 'No GPU detected'

# Check for TPU
try:
    import torch_xla.core.xla_model as xm
    print('✅ TPU Active:', xm.xla_device())
except Exception:
    print('No TPU detected (will use GPU or CPU)')

## 2. Install Packages & Clone Repo

In [ ]:
!pip install -q transformers[torch] datasets accelerate
!git clone https://github.com/youssefbouhaik/bulaq-ocr-transformer.git 2>/dev/null || echo "Repo already cloned"
%cd bulaq-ocr-transformer
# Decompress the sentence-level dataset
!gunzip -kf data/bulaq_sentence_pairs.jsonl.gz 2>/dev/null || echo "Already decompressed"

## 3. Load & Inspect Sentence-Level Dataset

In [ ]:
import json
from datasets import Dataset

data_path = 'data/bulaq_sentence_pairs.jsonl'
pairs = []
with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            src = row.get('corrupt_ocr', '').strip()
            tgt = row.get('ground_truth', '').strip()
            if src and tgt and src != tgt:
                pairs.append({'input_text': src, 'target_text': tgt})

print(f'✅ Loaded {len(pairs):,} sentence-level correction pairs!')

# Show sample pairs
for i in range(5):
    print(f"\n{i+1}. ❌ OCR:   '{pairs[i]['input_text'][:80]}...'")
    print(f"   ✅ Truth: '{pairs[i]['target_text'][:80]}...'")

# Show length statistics
src_lens = [len(p['input_text']) for p in pairs]
tgt_lens = [len(p['target_text']) for p in pairs]
print(f'\nSource lengths: min={min(src_lens)}, median={sorted(src_lens)[len(src_lens)//2]}, max={max(src_lens)}')
print(f'Target lengths: min={min(tgt_lens)}, median={sorted(tgt_lens)[len(tgt_lens)//2]}, max={max(tgt_lens)}')

# Split 90% train, 10% validation
raw_ds = Dataset.from_list(pairs).train_test_split(test_size=0.1, seed=42)
train_ds = raw_ds['train']
val_ds = raw_ds['test']
print(f'\nTrain: {len(train_ds):,} | Val: {len(val_ds):,}')

## 4. Initialize ByT5 Model & Tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = 'google/byt5-small'
print(f'Loading {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Detect device
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
except Exception:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)
print(f'✅ Model loaded on {device}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Tokenize Dataset

In [ ]:
max_src_len = 256
max_tgt_len = 256

def preprocess(batch):
    inputs = tokenizer(
        batch['input_text'],
        max_length=max_src_len,
        padding='max_length',
        truncation=True
    )
    targets = tokenizer(
        text_target=batch['target_text'],
        max_length=max_tgt_len,
        padding='max_length',
        truncation=True
    )
    labels = [
        [(t if t != tokenizer.pad_token_id else -100) for t in target]
        for target in targets['input_ids']
    ]
    inputs['labels'] = labels
    return inputs

print('Tokenizing train set...')
tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=['input_text', 'target_text'])
print('Tokenizing validation set...')
tokenized_val = val_ds.map(preprocess, batched=True, remove_columns=['input_text', 'target_text'])
print(f'✅ Tokenized: {len(tokenized_train):,} train, {len(tokenized_val):,} val')

## 6. Training Configuration & Execution
*With an A100/TPU and BF16, 3 epochs takes ~15-20 minutes on 69K pairs.*

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import inspect

# Detect precision
use_bf16 = False
use_fp16 = False
if torch.cuda.is_available():
    use_bf16 = hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported()
    use_fp16 = not use_bf16

training_args = Seq2SeqTrainingArguments(
    output_dir='./bulaq_byt5_checkpoints',
    optim='adamw_torch',
    learning_rate=3e-4,           # ⚡ FIXED: Stable LR for ByT5 (was 1e-3)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    warmup_steps=200,             # ⚡ FIXED: Gentle warmup
    weight_decay=0.01,
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=25,
    bf16=use_bf16,
    fp16=use_fp16,
    predict_with_generate=False,  # Faster training (no generation during eval)
    report_to='none',
)

# Set eval_strategy (handles different transformers versions)
if hasattr(training_args, 'eval_strategy'):
    training_args.eval_strategy = 'epoch'
elif hasattr(training_args, 'evaluation_strategy'):
    training_args.evaluation_strategy = 'epoch'

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Handle transformers API changes (processing_class vs tokenizer)
trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': tokenized_train,
    'eval_dataset': tokenized_val,
    'data_collator': data_collator,
}
trainer_sig = inspect.signature(Seq2SeqTrainer.__init__)
if 'processing_class' in trainer_sig.parameters:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_sig.parameters:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

# 🚀 START TRAINING
print('🚀 Starting ByT5 fine-tuning on 69,640 Bulaq sentence pairs...')
trainer.train()

## 7. Test The Trained Model

In [ ]:
import torch
from transformers import AutoTokenizer

# Use CPU for instant inference (avoids TPU XLA recompilation delays)
tokenizer = AutoTokenizer.from_pretrained('google/byt5-small')
eval_model = trainer.model.to('cpu')
eval_model.eval()

def correct_text(raw_ocr):
    inputs = tokenizer(
        raw_ocr, max_length=256, padding='max_length',
        truncation=True, return_tensors='pt'
    ).to('cpu')
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_length=256,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test cases: corrupted Bulaq sentences
test_cases = [
    'بلذنى ايها الماك السعيد ان ملكا من ملوك ساسان',
    'فلما سمع الملك شهريار هده الحكايه من شهرزاد',
    'والله لقدضاقت بى الارض لاجل غيبتك',
    'فنظر الى المرأه وهى قاعده على كرسى من الدهب',
    'قال لها يا سيدتى انا رجل غريب وقد تعبث من السفر',
    'ثم ان اخي دخل على الخليفه وقبل الارض بين يديه',
]

print('=== 🕌 BULAQ 1280 AH BYT5 HEALING TEST ===\n')
for raw in test_cases:
    healed = correct_text(raw)
    print(f'❌ Corrupt : {raw}')
    print(f'✅ Healed  : {healed}')
    print('-' * 60)

## 8. Save Trained Model

In [ ]:
# Save locally
output_dir = './bulaq_byt5_ocr_corrector'
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'✅ Model saved to {output_dir}')

# Optional: Save to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p '/content/drive/MyDrive/bulaq_byt5_ocr_corrector'
    !cp -r ./bulaq_byt5_ocr_corrector/* '/content/drive/MyDrive/bulaq_byt5_ocr_corrector/'
    print('✅ Model backed up to Google Drive!')
except Exception as e:
    print(f'Google Drive save skipped: {e}')